In [1]:
# ==============================================================================
# PyTorch Implementation: DenseNet201 + MobileNet Feature Concatenation Model
# ==============================================================================
# This script implements the dual-branch deep transfer learning architecture 
# (DenseNet201 + MobileNet) for 10-class camouflage image classification, 
# as described in the paper ("An approach of transfer learning and feature 
# concatenation for classification of camouflage images," Fig. 1).
#
# NOTE: You MUST replace the placeholder data loading logic 
# (SimpleCamouflageDataset) with your actual MultiDataset implementation
# from your original notebook, ensuring it correctly provides a 
# 10-class label (indices 0-9).
# ==============================================================================

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
import time

# --- 1. Configuration (Based on Paper & Notebook) ---

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Hyperparameters based on the paper's experimental setup
IMG_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 30
LR = 0.00001
NUM_CLASSES = 10 # 10 classes for COD10K/ERVA 1.0 species

# Seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device == "cuda": torch.cuda.manual_seed_all(SEED)

train_dir_cod = "/kaggle/input/cod10k/COD10K-v3/Train" 
test_dir_cod = "/kaggle/input/cod10k/COD10K-v3/Test"
train_dir_camo_cam = "/kaggle/input/camo-coco/CAMO_COCO/Camouflage"
train_dir_camo_noncam = "/kaggle/input/camo-coco/CAMO_COCO/Non_Camouflage"
testing_images_dir = "/kaggle/input/testing-dataset/Images"

# Fictional TXT files for demonstration (replace with actual path logic if needed)
# Since the paper implies using specific species, a custom label mapping would be needed,
# but for a complete pipeline, we keep the MultiDataset structure from the notebook.
ALL_ROOT_DIRS = [train_dir_cod, test_dir_cod, train_dir_camo_cam, train_dir_camo_noncam, testing_images_dir]

# --- 3. Data Utility Functions (Adapted from the notebook to support 10 classes) ---

def read_file_with_encoding(file_path, encodings=['utf-8', 'utf-8-sig', 'ISO-8859-1']):
    """Reads a file using a list of common encodings."""
    for encoding in encodings:
        try:
            with open(file_path, 'r', encoding=encoding) as f:
                return f.readlines()
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f"Unable to read {file_path} with any of the provided encodings.")
# --- 2. Data Transformation Pipelines ---

# Transformations based on common practice and the paper's mention of Random Rotation
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(15), 
    transforms.ToTensor(),
    # Standard ImageNet normalization for pre-trained models
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# --- 3. Dataset Placeholder (REPLACE THIS SECTION) ---

class SimpleCamouflageDataset(Dataset):
    """
    *** PLACEHOLDER CLASS - REPLACE WITH YOUR ACTUAL DATASET LOADER ***
    This class simulates loading data but uses dummy image paths and labels.
    You must replace the instantiation of this class with your working 
    MultiDataset or equivalent class that loads the 10-class data.
    """
    def __init__(self, data_list, transform=None):
        self.data_list = data_list
        self.transform = transform

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        img_path, label = self.data_list[idx]
        
        # In a real environment, you'd load the image here:
        # img = Image.open(img_path).convert("RGB")
        
        # Using a dummy tensor for demonstration purposes:
        # NOTE: If you use the dummy tensor, you should comment out the 
        # transforms.Normalize section in train_transform and val_transform.
        img = torch.rand(3, IMG_SIZE, IMG_SIZE) 
        
        # If you load real images, uncomment the transform line below:
        # if self.transform:
        #     img = self.transform(img)
            
        return img, torch.tensor(label, dtype=torch.long)

# --- 4. The Proposed DenseNet201 + MobileNet Model (Fig. 1 Implementation) ---

class DenseNetMobileNetFusion(nn.Module):
    """
    Implements the concatenation-based dual-branch TL architecture 
    DenseNet201 + MobileNet, based strictly on the paper's flow chart (Fig. 1).
    """
    def __init__(self, num_classes=10, freeze_weights=True):
        super().__init__()
        
        # --- Base Feature Extractor Layers ---
        
        # DenseNet201 Branch (Rich Hierarchical Features)
        densenet = models.densenet201(pretrained=True)
        self.densenet_features = densenet.features
        dense_out_channels = densenet.classifier.in_features # 1920
        
        # MobileNetV2 Branch (Lightweight, Generalizable Spatial Features)
        mobilenet = models.mobilenet_v2(pretrained=True)
        self.mobilenet_features = mobilenet.features
        mobile_out_channels = mobilenet.last_channel # 1280
        
        if freeze_weights:
            for param in self.densenet_features.parameters():
                param.requires_grad = False
            for param in self.mobilenet_features.parameters():
                param.requires_grad = False
            
        # --- Feature Refinement Heads (Per Branch) ---
        
        # DenseNet201 Head: GlobalAvgPool -> Dense 512 -> Drop 0.2 -> Dense 128 -> Drop 0.5 -> Dense 10
        self.densenet_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), 
            nn.Flatten(),
            nn.Linear(dense_out_channels, 512), 
            nn.ReLU(),
            nn.Dropout(0.2), 
            nn.Linear(512, 128), 
            nn.ReLU(),
            nn.Dropout(0.5), 
            nn.Linear(128, num_classes), # Intermediate output (10 features)
        )
        
        # MobileNet Head: GlobalAvgPool -> Dense 512 -> Drop 0.2 -> Dense 128 -> Drop 0.5 -> Dense 10
        self.mobilenet_head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), 
            nn.Flatten(),
            nn.Linear(mobile_out_channels, 512), 
            nn.ReLU(),
            nn.Dropout(0.2), 
            nn.Linear(512, 128), 
            nn.ReLU(),
            nn.Dropout(0.5), 
            nn.Linear(128, num_classes), # Intermediate output (10 features)
        )
        
        # --- Concatenation and Final Classification Head ---
        
        # Concatenate: Input size is 10 (DenseNet) + 10 (MobileNet) = 20
        # Final Head: Dense 512 -> Drop 0.2 -> Dense 128 -> Drop 0.5 -> Dense 10 (Softmax)
        self.classifier_head = nn.Sequential(
            nn.Linear(num_classes * 2, 512), 
            nn.ReLU(),
            nn.Dropout(0.2), 
            nn.Linear(512, 128), 
            nn.ReLU(),
            nn.Dropout(0.5), 
            nn.Linear(128, num_classes), 
            # Softmax is explicitly shown in the paper's diagram (Fig. 1).
            nn.Softmax(dim=1) 
        )

    def forward(self, x):
        d_feats = self.densenet_features(x)
        m_feats = self.mobilenet_features(x)
        
        d_out = self.densenet_head(d_feats)
        m_out = self.mobilenet_head(m_feats)
        
        # Feature Fusion (Concatenate)
        combined_features = torch.cat((d_out, m_out), dim=1)
        
        # Final Classification
        output = self.classifier_head(combined_features)
        
        return output

# --- 5. Training and Validation Loop ---

def train_and_validate_model(model, train_loader, val_loader, criterion, optimizer, epochs, device):
    best_val_accuracy = 0.0
    
    print("\n--- Starting Training ---")
    
    for epoch in range(epochs):
        start_time = time.time()
        
        # Training Phase
        model.train()
        train_loss = 0.0
        
        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            
            outputs = model(inputs)
            # Use Softmax output from the model (or adjust loss function if model output is logits)
            loss = criterion(outputs, labels) 
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            
        train_loss /= len(train_loader.dataset)
        
        # Validation Phase
        model.eval()
        corrects = 0
        total = 0
        val_loss = 0.0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                
                # Get the class with the highest probability
                _, preds = torch.max(outputs, 1)
                corrects += torch.sum(preds == labels.data)
                total += labels.size(0)

        val_accuracy = corrects.double() / total
        val_loss /= len(val_loader.dataset)
        
        epoch_time = time.time() - start_time
        
        print(f'Epoch {epoch+1}/{epochs} | Time: {epoch_time:.2f}s')
        print(f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy:.4f}')

        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            # torch.save(model.state_dict(), 'best_fusion_model.pth')
            # print("--- Saved best model state ---")

# --- 6. Execution Block ---

# 1. Initialize the Model
model = DenseNetMobileNetFusion(num_classes=NUM_CLASSES).to(device)

# 2. Define Loss and Optimizer (Adam optimizer and Cross-Entropy Loss)
criterion = nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# 3. Create DataLoaders (!!! ACTION REQUIRED: REPLACE THIS WITH YOUR ACTUAL DATA LOADING !!!)
# This dummy section MUST be replaced by your notebook's logic that correctly 
# loads and splits the COD10K and ERVA 1.0 data into 10 classes.

# DUMMY DATA CREATION (80% train / 20% val split, 100 total samples)
dummy_data = [(os.path.join(testing_images_dir, f'dummy_{i}.jpg'), random.randint(0, 9)) for i in range(100)]
train_data, val_data = train_test_split(dummy_data, test_size=0.2, random_state=SEED)

train_ds = SimpleCamouflageDataset(train_data, transform=train_transform)
val_ds = SimpleCamouflageDataset(val_data, transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print("\n" + "="*50)
print("             MODEL INITIALIZED SUCCESSFULLY")
print("="*50)
print(f"Model: DenseNet201 + MobileNet Fusion")
print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.2f} Million")
print(f"Total model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f} Million (most are frozen)")
print(f"Learning Rate: {LR}, Epochs: {EPOCHS}, Batch Size: {BATCH_SIZE}")
print(f"Train samples: {len(train_ds)}, Validation samples: {len(val_ds)} (DUMMY)")
print("="*50)

# 4. Run Training
train_and_validate_model(model, train_loader, val_loader, criterion, optimizer, EPOCHS, device)

Device: cuda


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet201_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet201_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/densenet201-c1103571.pth" to /root/.cache/torch/hub/checkpoints/densenet201-c1103571.pth
100%|██████████| 77.4M/77.4M [00:00<00:00, 213MB/s]
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may 


             MODEL INITIALIZED SUCCESSFULLY
Model: DenseNet201 + MobileNet Fusion
Total trainable parameters: 1.85 Million
Total model parameters: 22.17 Million (most are frozen)
Learning Rate: 1e-05, Epochs: 30, Batch Size: 8
Train samples: 80, Validation samples: 20 (DUMMY)

--- Starting Training ---
Epoch 1/30 | Time: 1.86s
Train Loss: 2.3022 | Val Loss: 2.3022 | Val Acc: 0.1000
Epoch 2/30 | Time: 0.66s
Train Loss: 2.3025 | Val Loss: 2.3022 | Val Acc: 0.1000
Epoch 3/30 | Time: 0.67s
Train Loss: 2.3025 | Val Loss: 2.3023 | Val Acc: 0.1000
Epoch 4/30 | Time: 0.66s
Train Loss: 2.3022 | Val Loss: 2.3023 | Val Acc: 0.1000
Epoch 5/30 | Time: 0.65s
Train Loss: 2.3024 | Val Loss: 2.3021 | Val Acc: 0.1000
Epoch 6/30 | Time: 0.66s
Train Loss: 2.3025 | Val Loss: 2.3021 | Val Acc: 0.1000
Epoch 7/30 | Time: 0.65s
Train Loss: 2.3026 | Val Loss: 2.3022 | Val Acc: 0.1000
Epoch 8/30 | Time: 0.66s
Train Loss: 2.3028 | Val Loss: 2.3022 | Val Acc: 0.1000
Epoch 9/30 | Time: 0.65s
Train Loss: 2.3029 | V

# Dual branch 